In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Jahangirpuri, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,282.00,445.70,40.55,56.00,62.70,38.43,18.23,1.90,7.82,3.73,14.24,87.37,0.44,186.89,975.07,12.29,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,228.17,370.65,29.87,39.89,45.47,42.50,16.99,1.66,13.94,2.26,8.58,87.63,0.51,201.21,975.21,12.59,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,338.36,523.51,75.86,50.94,88.82,40.19,18.54,2.70,12.64,3.94,16.96,91.16,0.33,182.16,975.11,13.47,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,276.75,425.62,43.27,57.38,65.72,39.49,14.59,1.99,14.03,4.13,13.83,92.96,0.45,168.90,975.00,13.34,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,202.62,341.92,26.50,50.54,48.43,38.54,16.80,1.41,19.45,2.20,6.49,86.50,0.58,162.31,975.25,13.42,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,376.64,626.29,38.56,98.83,83.92,16.24,30.25,3.64,10.74,2.85,16.41,54.79,0.30,326.31,982.00,17.56,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,343.02,559.41,26.45,102.60,76.07,6.26,38.71,2.76,16.05,2.91,11.80,56.13,0.30,326.48,982.00,17.41,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,262.38,462.02,34.06,103.79,82.87,17.44,27.24,2.06,31.59,2.47,7.39,55.23,0.30,326.38,982.00,17.29,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,295.96,509.96,50.67,108.83,99.05,46.80,25.80,2.19,37.69,2.78,9.21,55.14,0.30,321.65,982.00,17.18,0.0,0.0


In [4]:
df.shape
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 20 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   From Date  320 non-null    object 
 1   To Date    320 non-null    object 
 2   PM2.5      320 non-null    float64
 3   PM10       320 non-null    float64
 4   NO         320 non-null    float64
 5   NO2        320 non-null    float64
 6   NOx        320 non-null    float64
 7   NH3        320 non-null    float64
 8   SO2        320 non-null    float64
 9   CO         320 non-null    float64
 10  Ozone      320 non-null    float64
 11  Benzene    320 non-null    float64
 12  Toluene    320 non-null    float64
 13  RH         320 non-null    float64
 14  WS         317 non-null    float64
 15  WD         314 non-null    float64
 16  BP         320 non-null    float64
 17  AT         320 non-null    float64
 18  RF         320 non-null    float64
 19  TOT-RF     320 non-null    float64
dtypes: float64

In [5]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 20)


In [6]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 0
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
NH3          0
SO2          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [7]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")

    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [8]:


# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (320, 20)
          From Date           To Date   PM2.5    PM10      NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00   88.89  445.70  40.550  56.00  62.70   
1  02-01-2025 00:00  03-01-2025 00:00  228.17  370.65  29.870  39.89  45.47   
2  03-01-2025 00:00  04-01-2025 00:00   88.89  523.51  18.225  50.94  88.82   
3  04-01-2025 00:00  05-01-2025 00:00   88.89  425.62  43.270  57.38  65.72   
4  05-01-2025 00:00  06-01-2025 00:00  202.62  341.92  26.500  50.54  48.43   

     NH3    SO2    CO  Ozone  Benzene  Toluene     RH    WS      WD      BP  \
0  38.43  18.23  1.90   7.82     3.73    14.24  87.37  0.44  186.89  975.07   
1  42.50  16.99  1.66  13.94     2.26     8.58  87.63  0.51  201.21  975.21   
2  40.19  18.54  2.70  12.64     3.94    16.96  91.16  0.33  182.16  975.11   
3  39.49  14.59  1.99  14.03     4.13    13.83  92.96  0.45  168.90  975.00   
4  38.54  16.80  1.41  19.45     2.20     6.49  86.50  0.58  162.31  975.25   

      AT   RF  TOT-RF  
0  

In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [10]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,NH3,SO2,CO,Ozone,Benzene,Toluene,RH,WS,WD,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,-0.076433,1.918006,1.892128,0.648508,1.026362,0.335145,1.257029,0.792680,-1.329823,0.462018,-0.200060,1.460037,-0.800969,0.073333,-0.214385,-2.286488,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,2.936579,1.233123,0.961023,-0.015678,0.232301,0.689908,1.023223,0.389569,-1.049420,-0.016074,-0.443590,1.477231,-0.503428,0.813313,0.056392,-2.235933,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.076433,2.628076,-0.054214,0.439893,2.230127,0.488556,1.315481,2.136383,-1.108982,0.530316,-0.083028,1.710682,-1.268533,-0.171087,-0.137020,-2.087639,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.076433,1.734762,2.129264,0.705403,1.165542,0.427541,0.570694,0.943846,-1.045296,0.592111,-0.217701,1.829722,-0.758463,-0.856291,-0.349773,-2.109546,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.383862,0.970942,0.667219,0.423402,0.368716,0.344734,0.987397,-0.030338,-0.796965,-0.035588,-0.533516,1.402501,-0.205888,-1.196827,0.133756,-2.096065,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,-0.076433,-0.137053,1.718636,2.414312,2.004306,-1.599054,-0.079817,-0.164709,-1.196036,0.175813,-0.106693,-0.694585,-1.396050,-0.240073,0.230462,-1.398411,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,-0.076433,2.955688,0.662860,2.569742,1.642531,-2.468964,-0.079817,2.237161,-0.952744,0.195327,-0.305045,-0.605967,-1.396050,-0.240073,0.230462,-1.423688,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,-0.076433,2.066937,1.326316,2.618804,1.955915,-1.494456,2.955899,1.061420,-0.240740,0.052225,-0.494792,-0.665487,-1.396050,-0.240073,0.230462,-1.443910,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,-0.076433,2.504423,2.774412,2.826594,2.701586,1.064719,2.684382,1.279772,0.038747,0.153047,-0.416484,-0.671439,-1.396050,-0.240073,0.230462,-1.462447,0.0,0.0


In [11]:
df.to_excel('jahangirpur2025.xlsx', index=False)